# 🚀 Server Ummu NLP Lab — Google Colab

Notebook ini meluncurkan server **Ummu NLP Lab** di Google Colab dengan akselerasi GPU dan mengeksposnya ke internet menggunakan **Ngrok Static Domain**.

| Komponen | Detail |
|---|---|
| **Runtime** | Google Colab GPU (T4/L4) |
| **Backend** | Flask pada port 5000 |
| **Tunnel** | Ngrok Static Domain |
| **URL Permanen** | `https://gotten-kinsman-drained.ngrok-free.dev` |

> ⚠️ **Prasyarat:** Pastikan runtime Colab disetel ke **GPU** (`Runtime → Change runtime type → T4 GPU`).

Jalankan setiap sel di bawah ini secara berurutan dari atas ke bawah.

---

### Langkah 1 — Verifikasi GPU & Pasang Dependensi

In [ ]:
import subprocess, sys

# --- Verifikasi GPU ---
gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                           capture_output=True, text=True)
if gpu_check.returncode == 0:
    gpu_info = gpu_check.stdout.strip()
    print(f"✅ GPU terdeteksi: {gpu_info}")
else:
    print("⚠️  GPU tidak terdeteksi. Buka Runtime → Change runtime type → pilih T4 GPU.")
    print("    Server tetap bisa berjalan, tetapi training IndoBERT akan sangat lambat.")

# --- Pasang dependensi ---
print("\nMemasang dependensi Python...")
!pip install -q flask flask-cors pyngrok psutil pynvml scikit-learn \
    transformers accelerate torch pandas numpy requests google-auth python-dotenv

print("\n✅ Seluruh dependensi berhasil dipasang.")

### Langkah 2 — Unduh Kode Sumber dari GitHub

In [ ]:
import os

REPO_URL = "https://github.com/ummulfarihah/nlp_experimen_lab.git"
REPO_DIR = "nlp_experimen_lab"

if os.path.exists(f"/content/{REPO_DIR}"):
    print(f"Direktori '{REPO_DIR}' sudah ada, memperbarui kode...")
    %cd /content/{REPO_DIR}
    !git pull
else:
    print(f"Mengunduh repositori dari GitHub...")
    %cd /content
    !git clone {REPO_URL}
    %cd /content/{REPO_DIR}

print(f"\n✅ Kode sumber siap di: {os.getcwd()}")

### Langkah 3 — Hubungkan Tunnel Ngrok (Static Domain)

In [ ]:
from pyngrok import ngrok
import getpass

STATIC_DOMAIN = "gotten-kinsman-drained.ngrok-free.dev"

# Minta authtoken secara aman (tidak tersimpan di notebook)
NGROK_AUTHTOKEN = getpass.getpass("Masukkan Ngrok Authtoken Anda: ")

ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()  # Bersihkan tunnel lama jika ada

# Buka tunnel ke port 5000 dengan static domain
try:
    tunnel = ngrok.connect(5000, domain=STATIC_DOMAIN)
except Exception:
    try:
        tunnel = ngrok.connect(5000, hostname=STATIC_DOMAIN)
    except Exception:
        print("⚠️  Static domain gagal, menggunakan URL dinamis...")
        tunnel = ngrok.connect(5000)

public_url = tunnel.public_url

print()
print("=" * 60)
print("  🌐 TUNNEL NGROK BERHASIL TERHUBUNG")
print("=" * 60)
print(f"  Akses web app di: {public_url}")
print("=" * 60)

### Langkah 4 — Jalankan Server Flask

Sel ini akan terus berjalan selama server aktif. Untuk menghentikan server, klik tombol **■ Stop** pada sel ini.

In [ ]:
!python app.py